In [1]:
import pandas as pd


data = {
    "City": ["Jaipur", "Agra", "Udaipur", "Rishikesh", "Manali", "Goa", "Varanasi", "Jodhpur", "Darjeeling", "Shimla"],
    "State": ["Rajasthan", "Uttar Pradesh", "Rajasthan", "Uttarakhand", "Himachal Pradesh", "Goa", "Uttar Pradesh", "Rajasthan", "West Bengal", "Himachal Pradesh"],
    "Latitude": [26.9124, 27.1767, 24.5854, 30.0869, 32.2432, 15.2993, 25.3176, 26.2389, 27.0360, 31.1048],
    "Longitude": [75.7873, 78.0081, 73.7125, 78.2676, 77.1892, 74.1240, 82.9739, 73.0243, 88.2627, 77.1734],
    "Rating": [4.5, 4.3, 4.6, 4.7, 4.8, 4.4, 4.5, 4.3, 4.6, 4.7],
    "Popularity": [950, 1200, 850, 700, 1100, 1300, 900, 800, 750, 1000]
}

df = pd.DataFrame(data)
df.to_csv("travel_dataset.csv", index=False)


In [2]:
import math
def simple_distance(lat1, lon1, lat2, lon2):
    km_per_deg = 111
    x = (lat2 - lat1) * km_per_deg
    y = (lon2 - lon1) * km_per_deg * math.cos(math.radians(lat1))
    return math.sqrt(x**2 + y**2)

In [3]:
def rank_destinations(source_city, top_n=5):
    if source_city not in df['City'].values:
        return f"{source_city} not found in dataset."
    
    src = df[df['City'] == source_city].iloc[0]
    
    # Compute distances
    df['Distance'] = df.apply(
        lambda x: simple_distance(src['Latitude'], src['Longitude'], x['Latitude'], x['Longitude']),
        axis=1
    )
    df['NormRating'] = df['Rating'] / df['Rating'].max()
    df['NormPopularity'] = df['Popularity'] / df['Popularity'].max()
    df['NormDistance'] = df['Distance'] / df['Distance'].max()
    df['Score'] = 0.4*df['NormRating'] + 0.3*df['NormPopularity'] + 0.3*(1 - df['NormDistance'])
    
    recommendations = df[df['City'] != source_city].sort_values(by='Score', ascending=False)
    
    return recommendations[['City', 'State', 'Distance', 'Rating', 'Popularity', 'Score']].head(top_n)

In [4]:
cities_to_test = ["Jaipur", "Manali", "Rishikesh"]
for city in cities_to_test:
    print(f"\nTop destinations from {city}:")
    print(rank_destinations(city))


Top destinations from Jaipur:
      City             State    Distance  Rating  Popularity     Score
1     Agra     Uttar Pradesh  221.760914     4.3        1200  0.884062
4   Manali  Himachal Pradesh  607.770491     4.8        1100  0.813540
9   Shimla  Himachal Pradesh  485.158591     4.7        1000  0.810435
2  Udaipur         Rajasthan  329.985522     4.6         850  0.803309
7  Jodhpur         Rajasthan  283.512015     4.3         800  0.777499

Top destinations from Manali:
        City             State    Distance  Rating  Popularity     Score
9     Shimla  Himachal Pradesh  126.371106     4.7        1000  0.902510
1       Agra     Uttar Pradesh  567.612178     4.3        1200  0.845759
3  Rishikesh       Uttarakhand  259.881279     4.7         700  0.812229
0     Jaipur         Rajasthan  606.179402     4.5         950  0.798652
2    Udaipur         Rajasthan  910.530275     4.6         850  0.735920

Top destinations from Rishikesh:
       City             State    Distanc